## Pipeline: hybrid_metadata → human_scanpaths JSON

Convierte los datos de hybrid_metadata al formato JSON que necesita `extract_human_maps.py`.

**Inputs**
- `evts (UBA)`: `hybrid_metadata/Hybrid_ana_out/metadata/S*_full_metadata.csv`
- `evts (UON)`: `hybrid_metadata/UON_ana_out/metadata/*_full_metadata.csv`
- `bh_data (UBA)`: `hybrid_metadata/UBA/*/NN_hybrid_search_builder_code_*.csv`
- `bh_data (UON)`: `hybrid_metadata/UON/*_hybrid_search_builder_code_*.csv`

**Output**: `hybrid_metadata/human_scanpaths/*_scanpaths.json`

---

### Dependencia con datos de HSEM

| Dato | Fuente | Necesario para JSON | Necesario para modelo |
|------|--------|---------------------|-----------------------|
| `item_pos.csv` | `trials_properties.json` (HSEM) | ⚠️ Solo para `target_bbox` | No (el modelo lo lee de `trials_properties.json` directamente) |
| `ontarget`/`ondistractor` | `item_pos.csv` | No | No |
| `target_bbox` en JSON | `item_pos.csv` | Opcional | **No** — `extract_human_maps.py` ignora este campo del JSON |

**Conclusión**: el JSON es funcional para `extract_human_maps.py` aunque `target_bbox` sea `null`.  
El notebook usa `item_pos.csv` si ya existe en `hybrid_metadata/` — sin leer `trials_properties.json`.  
Si `item_pos.csv` no existe, se levanta una alerta y se sigue sin `target_bbox` ni `ontarget`/`ondistractor`.

In [9]:
import json
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.image as mpimg

SCREEN_SIZE     = [1920, 1080]
EXPERIMENT_PATH = '../Datasets/HSEM/images/'
ITEM_POS_CSV    = Path('../hybrid_metadata/item_pos.csv')
OUTPUT_DIR      = Path('../hybrid_metadata/output')
SCANPATHS_OUT   = Path('../hybrid_metadata/human_scanpaths')

IMAGE_HEIGHT   = 1024
IMAGE_WIDTH    = 1280
SCREEN_HEIGHT  = 1080
SCREEN_WIDTH   = 1920
RECEPTIVE_SIZE = 68
XIM            = 320   # offset pantalla→imagen (x)
YIM            = 28    # offset pantalla→imagen (y)

OUTPUT_DIR.mkdir(exist_ok=True)
SCANPATHS_OUT.mkdir(exist_ok=True)

### 1. Verificación de prerrequisitos

Chequea si `item_pos.csv` existe. Si no, las Secciones 2–3 (generación de output CSVs) funcionarán
en modo reducido: `ontarget`/`ondistractor` quedarán en NaN. El JSON de Sección 4 sigue siendo válido.

In [10]:
if ITEM_POS_CSV.exists():
    item_pos_df = pd.read_csv(ITEM_POS_CSV)
    bbox_lookup = {
        row['stm']: {
            'pos_x': row['pos_x'], 'pos_y': row['pos_y'],
            'height': row['height'], 'width': row['width'],
        }
        for _, row in item_pos_df.iterrows()
    }
    print(f'item_pos.csv cargado: {len(item_pos_df)} items | {item_pos_df["folder"].nunique()} imágenes')
    print('  [NOTA] item_pos.csv fue generado a partir de trials_properties.json (HSEM).')
    print('  Se usa para: ontarget/ondistractor en output CSVs y target_bbox en el JSON.')
    print('  extract_human_maps.py NO lee target_bbox del JSON — este campo es informativo.')
else:
    item_pos_df = None
    bbox_lookup = {}
    print('⚠️  ALERTA: item_pos.csv no encontrado en hybrid_metadata/')
    print()
    print('  Este archivo contiene las posiciones de cada estímulo en cada imagen.')
    print('  Fue generado originalmente desde Datasets/HSEM/trials_properties.json.')
    print()
    print('  Consecuencias de no tenerlo:')
    print('    - Secciones 2-3: ontarget/ondistractor quedarán NaN en los output CSVs.')
    print('    - Sección 4: target_bbox será null en el JSON.')
    print()
    print('  Impacto en extract_human_maps.py: NINGUNO.')
    print('  El modelo lee target_bbox de trials_properties.json directamente, no del JSON.')
    print()
    print('  Para regenerarlo (requiere trials_properties.json de HSEM):')
    print('    trials = json.load(open("Datasets/HSEM/trials_properties.json"))')
    print('    # extraer stim_name, stimulus_height, width, matched_column/row')
    print('    # guardar en hybrid_metadata/item_pos.csv')

item_pos.csv cargado: 1632 items | 102 imágenes
  [NOTA] item_pos.csv fue generado a partir de trials_properties.json (HSEM).
  Se usa para: ontarget/ondistractor en output CSVs y target_bbox en el JSON.
  extract_human_maps.py NO lee target_bbox del JSON — este campo es informativo.


### 2. Funciones auxiliares

- `start_stop_samples_trigg`: latencias de inicio/fin de cada fase por trigger.
- `closest_tuple`: objeto más cercano a una fijación dentro del umbral.

In [11]:
def start_stop_samples_trigg(evts, phase_name):
    phase_evts = evts[evts['type'] == phase_name].reset_index(drop=True)
    starts = list(phase_evts['latency'])
    stops  = list(phase_evts['latency'] + phase_evts['duration'])
    return starts, stops


def closest_tuple(item_positions, threshold, point):
    min_dist   = float('inf')
    closest_id = None
    for i, pos in enumerate(item_positions):
        dist = ((pos[0] - point[0])**2 + (pos[1] - point[1])**2) ** 0.5
        if dist < threshold and dist < min_dist:
            min_dist   = dist
            closest_id = i
    if closest_id is None:
        return False, None
    return True, closest_id

### 3. Agregar info de trial a los eventos

Enriquece los eventos de ET con columnas de trial, fase, MSS, y presencia/corrección del target.

**Dependencia de HSEM**: `item_pos_df` (cargado en Sección 1). Solo afecta `ontarget`/`ondistractor`.  
Si `item_pos_df` es `None`, esas columnas quedan NaN pero el resto del pipeline funciona.

In [12]:
def add_trial_info_to_events(evts, bh_data, thr, item_pos_df=None):
    """
    item_pos_df : DataFrame de item_pos.csv.
                  Si None, ontarget/ondistractor/stm quedan NaN (sin dependencia HSEM).
    """
    threshold  = thr
    exp_path   = EXPERIMENT_PATH
    screensize = SCREEN_SIZE

    image_names  = bh_data['searchimage'].drop_duplicates() \
                        .str.split('cmp_', expand=True)[1] \
                        .str.split('.jpg', expand=True)[0].to_list()
    targets      = bh_data.loc[::6, ['st5']]
    target_files = targets['st5'].str.lstrip('memstim').str.lstrip('/')[:-1]

    emvs = {'fixation', 'saccade'}
    tr   = 0
    for col in ['trial', 'phase', 'mss', 'ontarget', 'ondistractor', 'present',
                'correct', 'stm', 'inrange']:
        evts[col] = np.nan

    msss       = list(bh_data.loc[::6, 'Nstim'])
    T_and_stim = bh_data.loc[5::6, ['st5_cat', 'st5', 'key_resp.corr']].dropna(how='all')
    T_and_stim.loc[T_and_stim.st5 == 'memstim/dog1962.png'] = \
        T_and_stim.loc[T_and_stim.st5 == 'memstim/dog1962.png'].replace({0: 1, 1: 0})
    presents = list((T_and_stim['st5_cat'] == 'T') & ~(T_and_stim['st5'] == 'memstim/dog1962.png'))
    corrects = list(T_and_stim['key_resp.corr'].dropna() == 1)

    cross1_start_samp, cross1_stop_samp = start_stop_samples_trigg(evts, 'cross1')
    mem_start_samp,    mem_stop_samp    = start_stop_samples_trigg(evts, 'mem')
    cross2_start_samp, cross2_stop_samp = start_stop_samples_trigg(evts, 'cross2')
    vs_start_samp,     vs_stop_samp     = start_stop_samples_trigg(evts, 'vs')

    missing_images = []
    skip_vs        = False

    for index, row in evts.iterrows():
        if evts.at[index, 'type'] == 'cross1':
            tr        += 1
            image_name = image_names[tr - 1]
            img_path   = exp_path + 'cmp_' + image_name + '.jpg'
            try:
                img              = mpimg.imread(img_path)
                xim              = (screensize[0] - img.shape[1]) / 2
                yim              = (screensize[1] - img.shape[0]) / 2
                height, width, _ = img.shape
                x_bounds         = [-threshold, width  + threshold]
                y_bounds         = [-threshold, height + threshold]

                if item_pos_df is not None:
                    trial_stims = item_pos_df[item_pos_df['folder'] == image_name]
                    records     = trial_stims.to_records(index=False)
                    positions   = [(record[6] + record[5] / 2, record[7] + record[4] / 2)
                                   for record in records]
                    if presents[tr - 1]:
                        tp = trial_stims[trial_stims['stm'] == target_files.iloc[tr - 1]][
                            ['height', 'width', 'pos_x', 'pos_y']
                        ].to_records(index=False)
                        target_pos = (tp[0][2] + tp[0][1] / 2, tp[0][3] + tp[0][0] / 2)
                    else:
                        target_pos = None
                else:
                    trial_stims = pd.DataFrame()
                    positions   = []
                    target_pos  = None

                skip_vs = False
            except FileNotFoundError:
                missing_images.append(image_name)
                trial_stims = pd.DataFrame()
                positions   = []
                target_pos  = None
                skip_vs     = True

        elif evts.at[index, 'type'] in emvs:
            evts.at[index, 'trial']   = tr
            evts.at[index, 'mss']     = msss[tr - 1]
            evts.at[index, 'present'] = presents[tr - 1]
            evts.at[index, 'correct'] = corrects[tr - 1]

            lat = evts.at[index, 'latency']
            if   cross1_start_samp[tr-1] < lat < cross1_stop_samp[tr-1]:
                evts.at[index, 'phase'] = 'cross1'
            elif mem_start_samp[tr-1] < lat < mem_stop_samp[tr-1]:
                evts.at[index, 'phase'] = 'mem'
            elif cross2_start_samp[tr-1] < lat < cross2_stop_samp[tr-1]:
                evts.at[index, 'phase'] = 'cross2'
            elif vs_start_samp[tr-1] < lat < vs_stop_samp[tr-1]:
                evts.at[index, 'phase'] = 'vs'
                if skip_vs:
                    continue
                point = (evts.at[index, 'fix_avgpos_x'] - xim,
                         evts.at[index, 'fix_avgpos_y'] - yim)
                if evts.at[index, 'type'] == 'fixation':
                    evts.at[index, 'inrange'] = (
                        x_bounds[0] < point[0] < x_bounds[1] and
                        y_bounds[0] < point[1] < y_bounds[1]
                    )
                if item_pos_df is not None:
                    flag, closest_id = closest_tuple(positions, threshold, point)
                    if not flag:
                        continue
                    evts.at[index, 'stm'] = trial_stims['stm'].iloc[closest_id]
                    if positions[closest_id] == target_pos:
                        evts.at[index, 'ontarget']     = True
                        evts.at[index, 'ondistractor'] = False
                    else:
                        evts.at[index, 'ontarget']     = False
                        evts.at[index, 'ondistractor'] = True

    fix         = evts[evts['type'] == 'fixation']
    vs_counts   = len(fix[fix['phase'] == 'vs'])
    mean_dur_vs = fix[fix['phase'] == 'vs']['duration'].mean()
    answer_acc  = 100 * sum(corrects) / len(corrects)

    total_captured  = int(sum((evts['ondistractor'] == True) | (evts['ontarget'] == True)))
    on_targets      = int(sum(evts['ontarget']     == True))
    on_distractors  = int(sum(evts['ondistractor'] == True))

    if missing_images:
        print(f'  ⚠ imágenes no encontradas ({len(missing_images)}): {missing_images}')
    if item_pos_df is None:
        print(f'  [AVISO] ontarget/ondistractor no calculados (item_pos.csv no disponible)')
    print(f'  Correct: {answer_acc:.1f}%  |  VS fix: {vs_counts}  |  '
          f'On target: {on_targets}  |  On distractor: {on_distractors}  |  '
          + (f'Captured: {100*total_captured/vs_counts:.1f}%' if vs_counts > 0 else 'Captured: N/A'))

    stats = [answer_acc, vs_counts, total_captured, on_targets,
             100 * total_captured / vs_counts if vs_counts > 0 else np.nan, mean_dur_vs]
    return evts, stats, missing_images

### 4. Generar output CSVs (agregar info de trial a eventos)

Si los archivos en `hybrid_metadata/output/` ya existen, se puede saltar esta sección
y pasar directamente a la Sección 5.

`THR` = umbral en píxeles para asignar una fijación a un objeto (solo relevante si `item_pos_df` está disponible).

In [13]:
THR = 50

existing = sorted(OUTPUT_DIR.glob('*.csv'))
if existing:
    print(f'Output CSVs ya existen ({len(existing)} archivos). Saltar si no se desea regenerar.')
    print('  Ejemplo:', existing[0].name)

all_missing = {}

# ---------- UBA ----------
for meta_path in sorted(glob.glob('../hybrid_metadata/Hybrid_ana_out/metadata/S*_full_metadata.csv')):
    subject = Path(meta_path).stem.replace('_full_metadata', '')
    num     = subject.replace('S', '')
    builder = glob.glob(f'../hybrid_metadata/UBA/{subject}/{num}_hybrid_search_builder_code_*.csv')
    if not builder:
        print(f'{subject}: sin builder_code — omitido')
        continue
    print(f'--- {subject} ---')
    evts    = pd.read_csv(meta_path)
    bh_data = pd.read_csv(builder[0], encoding='utf-8-sig')
    evts, stats, missing = add_trial_info_to_events(evts, bh_data, thr=THR, item_pos_df=item_pos_df)
    evts.to_csv(OUTPUT_DIR / f'{subject}_full_metadata.csv', index=False)
    if missing:
        all_missing[subject] = missing

# ---------- UON ----------
for meta_path in sorted(glob.glob('../hybrid_metadata/UON_ana_out/metadata/*_full_metadata.csv')):
    subject = Path(meta_path).stem.replace('_full_metadata', '')
    builder = glob.glob(f'../hybrid_metadata/UON/{subject}_hybrid_search_builder_code_*.csv')
    if not builder:
        builder = glob.glob(f'../hybrid_metadata/UON/{subject}_2_hybrid_search_builder_code_*.csv')
    if not builder:
        print(f'{subject}: sin builder_code — omitido')
        continue
    print(f'--- {subject} ---')
    evts    = pd.read_csv(meta_path)
    bh_data = pd.read_csv(builder[0], encoding='utf-8-sig')
    evts, stats, missing = add_trial_info_to_events(evts, bh_data, thr=THR, item_pos_df=item_pos_df)
    evts.to_csv(OUTPUT_DIR / f'{subject}_full_metadata.csv', index=False)
    if missing:
        all_missing[subject] = missing

if all_missing:
    all_imgs = sorted({img for imgs in all_missing.values() for img in imgs})
    print(f'\nIMÁGENES FALTANTES ({len(all_imgs)} únicas):')
    for img in all_imgs:
        sujetos = [s for s, imgs in all_missing.items() if img in imgs]
        print(f'  cmp_{img}.jpg  →  afecta: {sujetos}')

Output CSVs ya existen (45 archivos). Saltar si no se desea regenerar.
  Ejemplo: 117969_full_metadata.csv
--- S101 ---


/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'present'] = presents[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:79: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'correct'] = corrects[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:89: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'vs' has dtype incompatible with float64, please explicitly cast to a compatible dty

  ⚠ imágenes no encontradas (6): ['shelves_041_book_014', 'shelves_044_toys_003', 'shelves_045_toys_004', 'shelves_048_toys_007', 'building_031_person_017', 'shelves_043_toys_002']
  Correct: 64.3%  |  VS fix: 5461  |  On target: 189  |  On distractor: 965  |  Captured: 21.1%
--- S102 ---


/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'present'] = presents[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:79: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'correct'] = corrects[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:89: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'vs' has dtype incompatible with float64, please explicitly cast to a compatible d

  ⚠ imágenes no encontradas (6): ['shelves_043_toys_002', 'shelves_044_toys_003', 'shelves_045_toys_004', 'building_031_person_017', 'shelves_041_book_014', 'shelves_048_toys_007']
  Correct: 82.4%  |  VS fix: 4893  |  On target: 213  |  On distractor: 726  |  Captured: 19.2%
--- S103 ---


...

  ⚠ imágenes no encontradas (6): ['shelves_041_book_014', 'shelves_044_toys_003', 'building_031_person_017', 'shelves_045_toys_004', 'shelves_048_toys_007', 'shelves_043_toys_002']
  Correct: 82.9%  |  VS fix: 3446  |  On target: 86  |  On distractor: 459  |  Captured: 15.8%
--- 862513 ---


/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'present'] = presents[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:79: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'correct'] = corrects[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:89: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'vs' has dtype incompatible with float64, please explicitly cast to a compatible d

  ⚠ imágenes no encontradas (6): ['shelves_041_book_014', 'shelves_045_toys_004', 'shelves_048_toys_007', 'shelves_043_toys_002', 'shelves_044_toys_003', 'building_031_person_017']
  Correct: 82.4%  |  VS fix: 4431  |  On target: 159  |  On distractor: 800  |  Captured: 21.6%
--- 963607 ---


/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'present'] = presents[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:79: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evts.at[index, 'correct'] = corrects[tr - 1]
/var/folders/8f/d2fjq2v97mj17hpklrr_c6yw0000gn/T/ipykernel_16362/130606985.py:89: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'vs' has dtype incompatible with float64, please explicitly cast to a compatible dty

  ⚠ imágenes no encontradas (6): ['shelves_044_toys_003', 'shelves_045_toys_004', 'building_031_person_017', 'shelves_048_toys_007', 'shelves_043_toys_002', 'shelves_041_book_014']
  Correct: 81.4%  |  VS fix: 3541  |  On target: 146  |  On distractor: 750  |  Captured: 25.3%

IMÁGENES FALTANTES (6 únicas):
  cmp_building_031_person_017.jpg  →  afecta: ['S101', 'S102', 'S103', 'S105', 'S106', 'S107', 'S108', 'S109', 'S111', 'S113', 'S114', 'S115', 'S116', 'S117', 'S118', 'S119', '117969', '123082', '159640', '179678', '200877', '244108', '244389', '248301', '286470', '305138', '370500', '373490', '389622', '400297', '419760', '501896', '533569', '573661', '576470', '601753', '619958', '629959', '664304', '677251', '712871', '719396', '848643', '862513', '963607']
  cmp_shelves_041_book_014.jpg  →  afecta: ['S101', 'S102', 'S103', 'S105', 'S106', 'S107', 'S108', 'S109', 'S111', 'S113', 'S114', 'S115', 'S116', 'S117', 'S118', 'S119', '117969', '123082', '159640', '179678', '200877', '244

### 5. Conversión a human_scanpaths JSON

Convierte los CSVs de `hybrid_metadata/output/` al formato `{subject}_scanpaths.json`.

**Fuentes (todas en hybrid_metadata — sin leer HSEM)**:
- `output/*.csv` → fijaciones filtradas (`type==fixation`, `phase==vs`, `bad==0`, `inrange==True`, `present==True`)
- `bh_data` → imagen y target por trial (log PsychoPy)
- `item_pos.csv` (opcional, ya en `hybrid_metadata/`) → `target_bbox`

**`target_bbox`**: si `item_pos.csv` está disponible se calcula; si no, vale `null`.  
`extract_human_maps.py` **no lee este campo del JSON** — el modelo lo obtiene de `trials_properties.json` directamente (línea 157 de `extract_human_maps.py`).  
Puede ser útil para análisis propios pero no afecta al pipeline de extracción de mapas.

In [14]:
def parse_stim(raw):
    """'memstim/truck196.png' → 'truck196.png'. NaN o blank → None."""
    if pd.isna(raw):
        return None
    name = str(raw).split('/')[-1]
    return None if name == 'blank.png' else name


def build_trial_info(bh_data):
    """Lista de info por trial extraída del log PsychoPy (fuente: hybrid_metadata)."""
    n_trials = len(bh_data) // 6
    info = []
    for i in range(n_trials):
        row0 = bh_data.iloc[i * 6]
        row5 = bh_data.iloc[i * 6 + 5]

        image          = row0['searchimage']
        target_stim    = parse_stim(row0['st5'])
        target_present = (str(row0['st5_cat']) == 'T') and target_stim is not None
        target_found   = (row5['key_resp.corr'] == 1.0)

        memory_set = [
            s for col in ['st1', 'st2', 'st3', 'st4', 'st5']
            if (s := parse_stim(row0[col])) is not None
        ]

        info.append({
            'image':          image,
            'target_stim':    target_stim,
            'target_present': target_present,
            'target_found':   bool(target_found),
            'memory_set':     memory_set,
        })
    return info


def output_csv_to_scanpaths(subject_id, output_csv_path, bh_data_path,
                             bbox_lookup=None, dataset_label='Hybrid Analysis ET'):
    """
    bbox_lookup : dict {stm → {pos_x, pos_y, height, width}} desde item_pos.csv.
                  Si None o vacío, target_bbox será null en el JSON.
                  extract_human_maps.py no lee target_bbox del JSON.
    """
    df  = pd.read_csv(output_csv_path)
    bh  = pd.read_csv(bh_data_path, encoding='utf-8-sig')

    trial_info = build_trial_info(bh)

    fix = df[
        (df['type']    == 'fixation') &
        (df['phase']   == 'vs')       &
        (df['bad']     == 0)          &
        (df['inrange'] == True)       &
        (df['present'] == True)
    ].copy().sort_values(['trial', 'latency'])

    scanpaths = {}
    alerts    = []

    for trial_id, trial_df in fix.groupby('trial'):
        idx = int(trial_id) - 1

        if idx >= len(trial_info):
            alerts.append(f'trial {trial_id}: índice fuera de rango ({idx} >= {len(trial_info)})')
            continue

        info = trial_info[idx]

        if not info['target_present']:
            continue

        target_stim = info['target_stim']
        image       = info['image']

        if len(trial_df) == 0:
            alerts.append(f'trial {trial_id} ({image}): sin fijaciones válidas tras filtros')
            continue

        # target_bbox: desde item_pos.csv si disponible, else null
        if bbox_lookup and target_stim in bbox_lookup:
            b    = bbox_lookup[target_stim]
            bbox = [float(b['pos_y']), float(b['pos_x']),
                    float(b['pos_y'] + b['height']), float(b['pos_x'] + b['width'])]
        else:
            bbox = None  # no afecta a extract_human_maps.py

        scanpaths[image] = {
            'subject'           : subject_id,
            'dataset'           : dataset_label,
            'image_height'      : IMAGE_HEIGHT,
            'image_width'       : IMAGE_WIDTH,
            'screen_height'     : SCREEN_HEIGHT,
            'screen_width'      : SCREEN_WIDTH,
            'receptive_height'  : RECEPTIVE_SIZE,
            'receptive_width'   : RECEPTIVE_SIZE,
            'memory_set'        : info['memory_set'],
            'target_stim'       : target_stim,
            'target_found'      : info['target_found'],
            'target_present'    : True,
            'target_bbox'       : bbox,
            'X'                 : (trial_df['fix_avgpos_x'] - XIM).tolist(),
            'Y'                 : (trial_df['fix_avgpos_y'] - YIM).tolist(),
            'target_object'     : 'TBD',
            'max_fixations'     : None,
            'seen_order'        : [],
            'fix_proportion_mem': [],
        }

    for a in alerts:
        print(f'  [ALERT] {a}')

    return scanpaths


def save_scanpaths(scanpaths, subject_id, overwrite=False):
    out = SCANPATHS_OUT / f'{subject_id}_scanpaths.json'
    if out.exists() and not overwrite:
        print(f'  Ya existe {out.name}. Saltar o usar OVERWRITE=True.')
        return
    with open(out, 'w') as f:
        json.dump(scanpaths, f, indent=2)
    print(f'  Guardado: {out.name} ({len(scanpaths)} imágenes)')

In [15]:
OVERWRITE = True

# ---------- UBA ----------
print('=== Sujetos UBA ===')
for output_path in sorted(OUTPUT_DIR.glob('S*_full_metadata.csv')):
    subject = output_path.stem.replace('_full_metadata', '')
    num     = subject.replace('S', '')
    builder = glob.glob(f'../hybrid_metadata/UBA/{subject}/{num}_hybrid_search_builder_code_*.csv')
    if not builder:
        print(f'[ALERT] {subject}: sin bh_data — omitido')
        continue
    print(f'\n[{subject}]')
    sp = output_csv_to_scanpaths(subject, output_path, builder[0], bbox_lookup=bbox_lookup)
    if sp:
        save_scanpaths(sp, subject, overwrite=OVERWRITE)

# ---------- UON ----------
print('\n=== Sujetos UON ===')
for output_path in sorted(OUTPUT_DIR.glob('[0-9]*_full_metadata.csv')):
    subject = output_path.stem.replace('_full_metadata', '')
    builder = glob.glob(f'../hybrid_metadata/UON/{subject}_hybrid_search_builder_code_*.csv')
    if not builder:
        builder = glob.glob(f'../hybrid_metadata/UON/{subject}_2_hybrid_search_builder_code_*.csv')
    if not builder:
        print(f'[ALERT] {subject}: sin bh_data — omitido')
        continue
    print(f'\n[{subject}]')
    sp = output_csv_to_scanpaths(subject, output_path, builder[0], bbox_lookup=bbox_lookup)
    if sp:
        save_scanpaths(sp, subject, overwrite=OVERWRITE)

print('\nListo.')

=== Sujetos UBA ===

[S101]
  Guardado: S101_scanpaths.json (101 imágenes)

[S102]
  Guardado: S102_scanpaths.json (101 imágenes)

[S103]
  Guardado: S103_scanpaths.json (101 imágenes)

[S105]
  Guardado: S105_scanpaths.json (101 imágenes)

[S106]
  Guardado: S106_scanpaths.json (102 imágenes)

[S107]
  Guardado: S107_scanpaths.json (102 imágenes)

[S108]
  Guardado: S108_scanpaths.json (90 imágenes)

[S109]
  Guardado: S109_scanpaths.json (101 imágenes)

[S111]
  Guardado: S111_scanpaths.json (100 imágenes)

[S113]
  Guardado: S113_scanpaths.json (100 imágenes)

[S114]
  Guardado: S114_scanpaths.json (102 imágenes)

[S115]
  Guardado: S115_scanpaths.json (101 imágenes)

[S116]
  Guardado: S116_scanpaths.json (102 imágenes)

[S117]
  Guardado: S117_scanpaths.json (102 imágenes)

[S118]
  Guardado: S118_scanpaths.json (101 imágenes)

[S119]
  Guardado: S119_scanpaths.json (98 imágenes)

=== Sujetos UON ===

[117969]
  Guardado: 117969_scanpaths.json (102 imágenes)

[123082]
  Guardado: 

### 6. Validación: caso testigo

Muestra paso a paso la construcción de una entrada del JSON para S101,
verificando que la info proviene de bh_data y output CSV (sin HSEM).

In [17]:
SUBJECT_VAL  = 'S101'
OUTPUT_CSV   = OUTPUT_DIR / f'{SUBJECT_VAL}_full_metadata.csv'
BH_DATA_PATH = glob.glob(f'../hybrid_metadata/UBA/{SUBJECT_VAL}/101_hybrid_search_builder_code_*.csv')[0]

df_val = pd.read_csv(OUTPUT_CSV)
bh_val = pd.read_csv(BH_DATA_PATH, encoding='utf-8-sig')
sp_val = json.load(open(SCANPATHS_OUT / f'{SUBJECT_VAL}_scanpaths.json'))

trial_info_val = build_trial_info(bh_val)

fix_val = df_val[
    (df_val['type']    == 'fixation') &
    (df_val['phase']   == 'vs')       &
    (df_val['bad']     == 0)          &
    (df_val['inrange'] == True)       &
    (df_val['present'] == True)
].sort_values(['trial', 'latency'])

trial_correct   = fix_val[fix_val['correct'] == True ]['trial'].iloc[0]
trial_incorrect = fix_val[fix_val['correct'] == False]['trial'].iloc[0]


def show_case(trial_id, label):
    idx  = int(trial_id) - 1
    info = trial_info_val[idx]
    t_df = fix_val[fix_val['trial'] == trial_id]

    print('=' * 60)
    print(f'CASO: {label}  |  Trial {int(trial_id)}  |  {SUBJECT_VAL}')
    print('=' * 60)

    print('\n── 1. bh_data (fuente: hybrid_metadata) ──')
    row0 = bh_val.iloc[idx * 6]
    row5 = bh_val.iloc[idx * 6 + 5]
    print(f'  searchimage  : {row0["searchimage"]}')
    print(f'  st1-st5      : {[row0[c] for c in ["st1","st2","st3","st4","st5"]]}')
    print(f'  st5_cat      : {row0["st5_cat"]}')
    print(f'  key_resp.corr: {row5["key_resp.corr"]}')
    print(f'  Nstim        : {int(row0["Nstim"])}')

    print('\n── 2. build_trial_info() ──')
    for k, v in info.items():
        print(f'  {k:<16}: {v}')

    print('\n── 3. target_bbox ──')
    if bbox_lookup and info['target_stim'] in bbox_lookup:
        b    = bbox_lookup[info['target_stim']]
        bbox = [b['pos_y'], b['pos_x'], b['pos_y'] + b['height'], b['pos_x'] + b['width']]
        print(f'  Desde item_pos.csv: {bbox}  → [y_min, x_min, y_max, x_max]')
        print(f'  [NOTA] item_pos.csv es derivado de HSEM/trials_properties.json')
        print(f'  [NOTA] extract_human_maps.py NO usa este campo del JSON')
    else:
        print(f'  null (item_pos.csv no disponible — sin impacto en extract_human_maps.py)')

    print('\n── 4. Fijaciones filtradas (fuente: output CSV) ──')
    print(f'  Fijaciones: {len(t_df)}')
    print(t_df[['latency', 'fix_avgpos_x', 'fix_avgpos_y',
                'ontarget', 'ondistractor']].to_string(index=False))

    print('\n── 5. Entrada en el JSON ──')
    entry = sp_val.get(info['image'], {})
    for k, v in entry.items():
        if isinstance(v, list) and len(v) > 4:
            print(f'  {k:<18}: [{v[0]}, {v[1]}, ... ] (n={len(v)})')
        else:
            print(f'  {k:<18}: {v}')
    print()


show_case(trial_correct,   'CORRECTO')
show_case(trial_incorrect, 'INCORRECTO')

CASO: CORRECTO  |  Trial 1  |  S101

── 1. bh_data (fuente: hybrid_metadata) ──
  searchimage  : cmp_shelves_059_toys_018.jpg
  st1-st5      : ['memstim/blank.png', 'memstim/blank.png', 'memstim/blank.png', 'memstim/blank.png', 'memstim/truck196.png']
  st5_cat      : T
  key_resp.corr: 1.0
  Nstim        : 1

── 2. build_trial_info() ──
  image           : cmp_shelves_059_toys_018.jpg
  target_stim     : truck196.png
  target_present  : True
  target_found    : True
  memory_set      : ['truck196.png']

── 3. target_bbox ──
  Desde item_pos.csv: [84, 598, 133, 678]  → [y_min, x_min, y_max, x_max]
  [NOTA] item_pos.csv es derivado de HSEM/trials_properties.json
  [NOTA] extract_human_maps.py NO usa este campo del JSON

── 4. Fijaciones filtradas (fuente: output CSV) ──
  Fijaciones: 6
 latency  fix_avgpos_x  fix_avgpos_y ontarget ondistractor
 54055.0    799.259033    481.756317    False         True
 54110.0    810.220154    490.580231    False         True
 54216.0    592.137939    3